In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add
from tensorflow.keras.utils import to_categorical

# 1. Dataset & Tokenizer Setup
raw_captions = [
    "startseq a child running in the snow endseq",
    "startseq young boys racing on a snowy day endseq"
]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(raw_captions)
vocab_size = len(tokenizer.word_index) + 1
max_length = max(len(c.split()) for c in raw_captions)

# 2. Image Feature Extraction
def extract_vgg_features(image_path=None):
    if image_path:
        vgg = VGG16()
        extractor = Model(inputs=vgg.inputs, outputs=vgg.layers[-2].output)
        img = preprocess_input(np.expand_dims(img_to_array(load_img(image_path, target_size=(224, 224))), axis=0))
        return extractor.predict(img, verbose=0)
    return np.random.rand(1, 4096)  # Dummy vector for testing

# 3. Data Sequence Matrix Construction
def prepare_sequences(captions, feature_vector):
    X1, X2, y = [], [], []
    for caption in captions:
        seq = tokenizer.texts_to_sequences([caption])[0]
        for i in range(1, len(seq)):
            in_seq = pad_sequences([seq[:i]], maxlen=max_length, padding='post')[0]
            out_seq = to_categorical([seq[i]], num_classes=vocab_size)[0]
            X1.append(feature_vector[0])
            X2.append(in_seq)
            y.append(out_seq)
    return np.array(X1), np.array(X2), np.array(y)

# 4. Neural Network Architecture
def build_caption_model():
    # Image Feature Branch
    img_input = Input(shape=(4096,))
    img_dense = Dense(256, activation='relu')(Dropout(0.4)(img_input))

    # Text Sequence Branch
    text_input = Input(shape=(max_length,))
    text_emb = Embedding(vocab_size, 256, mask_zero=True)(text_input)
    text_lstm = LSTM(256)(Dropout(0.4)(text_emb))

    # Output Decoder
    merged = add([img_dense, text_lstm])
    decoder = Dense(256, activation='relu')(merged)
    output = Dense(vocab_size, activation='softmax')(decoder)

    model = Model(inputs=[img_input, text_input], outputs=output)
    model.compile(loss='categorical_crossentropy', optimizer='adam')
    return model

# 5. Caption Generation
def generate_caption(model, feature):
    in_text = 'startseq'
    for _ in range(max_length):
        seq = tokenizer.texts_to_sequences([in_text])[0]
        seq = pad_sequences([seq], maxlen=max_length, padding='post')
        pred_idx = np.argmax(model.predict([feature, seq], verbose=0))
        
        word = next((w for w, idx in tokenizer.word_index.items() if idx == pred_idx), None)
        if word is None or word == 'endseq':
            break
        in_text += ' ' + word
    return in_text.replace('startseq', '').strip()

# 6. Model Training & Testing
if __name__ == "__main__":
    img_features = extract_vgg_features()
    X1, X2, y = prepare_sequences(raw_captions, img_features)

    model = build_caption_model()
    model.fit([X1, X2], y, epochs=10, batch_size=32, verbose=0)

    caption = generate_caption(model, img_features)
    print("Generated Caption:", caption)